# Porn Detection

#####  на основе предоставленного набора тренировочных данных необходимо построить бинарный классификатор веб-страниц.

Подключим pandas для преобразования и считывания из csv и sklearn для обучения модели, метрик

In [ ]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

Считаем данные из файлов, удалим пустые строки

In [ ]:
train = pd.read_csv("train.csv")
train = train.dropna()
test = pd.read_csv("test.csv")

объединим url и title, сразу же приведём данные к нижнему регистру

In [ ]:
train = train[["url", "title", "label"]]
train["text"] = train["url"].str.lower() + ' ' + train["title"].str.lower()
train.head()

,url,title,label,text
0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0,m.kp.md экс-министр экономики молдовы - главе ...
1,www.kp.by,Эта песня стала известна многим телезрителям б...,0,www.kp.by эта песня стала известна многим теле...
2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0,fanserials.tv банши 4 сезон 2 серия бремя крас...
3,colorbox.spb.ru,Не Беси Меня Картинки,0,colorbox.spb.ru не беси меня картинки
4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0,tula-sport.ru в новомосковске сыграют следж-хо...


действительно, данные преобразовались

разделим данные на тренировочные и тестовые

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(train[["text"]], train["label"], 
                                                    test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((108246, 1), (27062, 1), (108246,), (27062,))

используем TfidVectorizer, задав ему следующие параметры:

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 4),
    max_features=30000,
    min_df=3,
    max_df=0.85,
    analyzer='char_wb',
    sublinear_tf=True,
    norm='l2',
    smooth_idf=False,
    strip_accents='unicode'
)

преобразуем обучающие тексты в числовое представление

In [ ]:
X_train_vect = vectorizer.fit_transform(x_train["text"])
X_test_vect = vectorizer.transform(x_test["text"])

обучим модель

In [ ]:
clf = LogisticRegression(penalty="l2")

In [ ]:
%%time
clf.fit(X_train_vect, y_train)
predicted = clf.predict(X_test_vect)

print("Precision: %.3f" % precision_score(y_test, predicted))
print("Recall: %.3f" % recall_score(y_test, predicted))
print("F1: %.3f" % f1_score(y_test, predicted))

Precision: 0.997
Recall: 0.946
F1: 0.971
CPU times: total: 812 ms
Wall time: 1.55 s


получили отличные параметры по времени работы и метрикам

проделаем всё то же самое на тренировочных (тестовых) данных

In [ ]:
test["text"] = test["url"] + ' ' + test["title"]
test["text"] = test["text"].str.lower()
X_test_vec = vectorizer.transform(test["text"])

pred = clf.predict(X_test_vec)

test["label"] = pred
test = test[["ID", "label"]]

test.to_csv("nlp.csv", index=False)

In [ ]:
test.head()

,ID,label
0,135309,0
1,135310,0
2,135311,0
3,135312,1
4,135313,0


получаем csv файлик, загружаем на kaggle